In [ ]:
# Run first to use NER
!python -m spacy download en_core_web_trf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 37.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Setup

### Fetch package

In [ ]:
# prompt: delete a specified directory

!rm -r /content/community_collection

rm: cannot remove '/content/community_collection': No such file or directory


In [ ]:
!git clone  https://github.com/gl0bsec/community_collection.git


Cloning into 'community_collection'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 126 (delta 61), reused 58 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 508.70 KiB | 2.49 MiB/s, done.
Resolving deltas: 100% (61/61), done.


### Install depndencies

In [ ]:
!pip install -r /content/community_collection/requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 19.8 MB/s eta 0:00:00


In [ ]:
!pip install pacmap -q
!pip install seaborn -q
!pip install gdelt -q
!pip install langchain_community -q
!pip install selenium -q
!pip install html2text -q
!pip install unstructured -q
# !pip install open-interpreter -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.4/787.4 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import pandas as pd
from typing import Dict, Optional, List, Tuple
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings


### Helper functions

In [ ]:
import numpy as np
from math import ceil
from hdbscan import HDBSCAN
import math

def extract_url_metadata(url: str, timeout: int = 10) -> Dict[str, Optional[str]]:
    """
    Extract metadata from a webpage URL including title, description, and other relevant information.
    """
    metadata = {
        'url': url,
        'title': None,
        'description': None,
        'keywords': None,
        'author': None,
        'site_name': None,
        'image': None,
        'favicon': None,
        'canonical_url': None,
        'language': None,
        'content_type': None,
        'status_code': None,
        'error': None
    }

    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        response = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
        metadata['status_code'] = response.status_code
        metadata['content_type'] = response.headers.get('content-type', '')

        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Extract title
        title_tag = soup.find('title')
        if title_tag:
            metadata['title'] = title_tag.get_text().strip()

        # Extract meta tags
        meta_tags = soup.find_all('meta')

        for tag in meta_tags:
            if tag.get('name', '').lower() == 'description':
                metadata['description'] = tag.get('content', '').strip()
            elif tag.get('name', '').lower() == 'keywords':
                metadata['keywords'] = tag.get('content', '').strip()
            elif tag.get('name', '').lower() == 'author':
                metadata['author'] = tag.get('content', '').strip()
            elif tag.get('name', '').lower() == 'language':
                metadata['language'] = tag.get('content', '').strip()
            elif tag.get('property', '').lower() == 'og:title' and not metadata['title']:
                metadata['title'] = tag.get('content', '').strip()
            elif tag.get('property', '').lower() == 'og:description' and not metadata['description']:
                metadata['description'] = tag.get('content', '').strip()
            elif tag.get('property', '').lower() == 'og:site_name':
                metadata['site_name'] = tag.get('content', '').strip()
            elif tag.get('property', '').lower() == 'og:image':
                metadata['image'] = tag.get('content', '').strip()
            elif tag.get('name', '').lower() == 'twitter:title' and not metadata['title']:
                metadata['title'] = tag.get('content', '').strip()
            elif tag.get('name', '').lower() == 'twitter:description' and not metadata['description']:
                metadata['description'] = tag.get('content', '').strip()
            elif tag.get('name', '').lower() == 'twitter:image' and not metadata['image']:
                metadata['image'] = tag.get('content', '').strip()

        # Extract canonical URL
        canonical_link = soup.find('link', {'rel': 'canonical'})
        if canonical_link:
            metadata['canonical_url'] = canonical_link.get('href', '').strip()

        # Extract favicon
        favicon_link = soup.find('link', {'rel': 'icon'}) or soup.find('link', {'rel': 'shortcut icon'})
        if favicon_link:
            favicon_href = favicon_link.get('href', '')
            if favicon_href:
                metadata['favicon'] = urljoin(url, favicon_href)

        # Extract language from html tag if not found in meta
        if not metadata['language']:
            html_tag = soup.find('html')
            if html_tag:
                metadata['language'] = html_tag.get('lang', '').strip()

        # Convert relative URLs to absolute
        if metadata['image'] and not metadata['image'].startswith(('http://', 'https://')):
            metadata['image'] = urljoin(url, metadata['image'])

        if metadata['canonical_url'] and not metadata['canonical_url'].startswith(('http://', 'https://')):
            metadata['canonical_url'] = urljoin(url, metadata['canonical_url'])

    except requests.exceptions.RequestException as e:
        metadata['error'] = f"Request error: {str(e)}"
    except Exception as e:
        metadata['error'] = f"Parsing error: {str(e)}"

    # Clean up empty values
    metadata = {k: v if v else None for k, v in metadata.items()}

    return metadata


def get_source_urls_with_metadata(df, actor1_code=None, actor2_code=None, geo_code=None,
                                 match_type='any', limit=None, show_events=True,
                                 extract_metadata=False, max_workers=5, delay=1.0,
                                 timeout=10, dataF=False):
    """
    Get source URLs with optional metadata extraction for events matching country code criteria.

    Parameters:
    - df: GDELT DataFrame
    - actor1_code: Country code for Actor1 (None = any country)
    - actor2_code: Country code for Actor2 (None = any country)
    - geo_code: Country code for event location (None = any location)
    - match_type: 'any' (OR) or 'all' (AND) for multiple conditions
    - limit: Maximum number of rows/URLs to return (None = all)
    - show_events: Whether to include event descriptions (True/False)
    - extract_metadata: Whether to extract webpage metadata (True/False)
    - max_workers: Number of concurrent threads for metadata extraction
    - delay: Delay between requests in seconds (to be respectful)
    - timeout: Request timeout in seconds
    - dataF: Return a DataFrame with full event details and metadata

    Returns:
    - DataFrame if dataF=True (includes metadata columns if extract_metadata=True)
    - List of tuples with metadata if extract_metadata=True
    - List of tuples (url, event_descriptions) if show_events=True
    - List of URLs if show_events=False
    """

    # Build conditions
    conditions = []

    if actor1_code:
        conditions.append(df['Actor1CountryCode'] == actor1_code)
    if actor2_code:
        conditions.append(df['Actor2CountryCode'] == actor2_code)
    if geo_code:
        conditions.append(df['ActionGeo_CountryCode'] == geo_code)

    # Apply conditions or use entire dataset if no filters specified
    if not conditions:
        print("No filters specified. Processing entire dataset...")
        filtered_df = df.copy()
    else:
        # Apply conditions based on match_type
        if match_type == 'any':
            mask = conditions[0]
            for condition in conditions[1:]:
                mask = mask | condition
        else:  # 'all'
            mask = conditions[0]
            for condition in conditions[1:]:
                mask = mask & condition

        # Filter dataframe
        filtered_df = df[mask].copy()

    # Check if we have any data to process
    if len(filtered_df) == 0:
        print("No events found matching the specified criteria.")
        if dataF:
            return pd.DataFrame()
        return []

    if dataF:
        # Aggregate GDELT variables by URL
        print("Aggregating GDELT events by URL...")

        # Convert SQLDATE to datetime first for aggregation
        filtered_df['SQLDATE_dt'] = pd.to_datetime(filtered_df['SQLDATE'], format='%Y%m%d')

        # Define aggregation functions for each column
        agg_dict = {
            'GoldsteinScale': ['mean', 'min', 'max', 'std', 'count'],  # Multiple stats for sentiment
            'Actor1Name': lambda x: ' | '.join(x.dropna().unique()[:5]),  # Top 5 unique actors
            'Actor2Name': lambda x: ' | '.join(x.dropna().unique()[:5]),  # Top 5 unique actors
            'Actor1CountryCode': lambda x: ' | '.join(x.dropna().unique()),  # All unique countries
            'Actor2CountryCode': lambda x: ' | '.join(x.dropna().unique()),  # All unique countries
            'ActionGeo_CountryCode': lambda x: ' | '.join(x.dropna().unique()),  # All unique locations
            'SQLDATE_dt': ['min', 'max'],  # Date range
            'EventDescription': lambda x: list(x.dropna().unique()) if 'EventDescription' in filtered_df.columns else []
        }

        # Perform aggregation
        result_df = filtered_df.groupby('SOURCEURL').agg(agg_dict).reset_index()

        # Flatten column names
        result_df.columns = ['_'.join(col).strip('_') if col[1] else col[0]
                           for col in result_df.columns.values]

        # Rename columns for clarity
        column_mapping = {
            'GoldsteinScale_mean': 'avg_goldstein_score',
            'GoldsteinScale_min': 'min_goldstein_score',
            'GoldsteinScale_max': 'max_goldstein_score',
            'GoldsteinScale_std': 'goldstein_score_std',
            'GoldsteinScale_count': 'event_count',
            'Actor1Name_<lambda>': 'actor1_names',
            'Actor2Name_<lambda>': 'actor2_names',
            'Actor1CountryCode_<lambda>': 'actor1_countries',
            'Actor2CountryCode_<lambda>': 'actor2_countries',
            'ActionGeo_CountryCode_<lambda>': 'event_locations',
            'SQLDATE_dt_min': 'first_event_date',
            'SQLDATE_dt_max': 'last_event_date',
            'EventDescription_<lambda>': 'event_descriptions'
        }

        # Apply column mapping
        for old_col, new_col in column_mapping.items():
            if old_col in result_df.columns:
                result_df = result_df.rename(columns={old_col: new_col})

        # Convert dates to string format
        if 'first_event_date' in result_df.columns:
            result_df['first_event_date'] = result_df['first_event_date'].dt.strftime('%Y-%m-%d')
        if 'last_event_date' in result_df.columns:
            result_df['last_event_date'] = result_df['last_event_date'].dt.strftime('%Y-%m-%d')

        # Add derived metrics
        if 'first_event_date' in result_df.columns and 'last_event_date' in result_df.columns:
            result_df['date_span_days'] = (
                pd.to_datetime(result_df['last_event_date']) -
                pd.to_datetime(result_df['first_event_date'])
            ).dt.days

        # Fill NaN standard deviations with 0 (single event cases)
        if 'goldstein_score_std' in result_df.columns:
            result_df['goldstein_score_std'] = result_df['goldstein_score_std'].fillna(0)

        # Sort by event count (most events first)
        if 'event_count' in result_df.columns:
            result_df = result_df.sort_values('event_count', ascending=False)

        # Apply limit if specified
        if limit:
            result_df = result_df.head(limit)

        # Extract metadata if requested
        if extract_metadata:
            print(f"Extracting metadata for {len(result_df)} URLs...")
            metadata_results = []

            def extract_with_delay(url):
                time.sleep(delay)
                return extract_url_metadata(url, timeout)

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                future_to_url = {executor.submit(extract_with_delay, url): url
                               for url in result_df['SOURCEURL'].unique()}

                for future in as_completed(future_to_url):
                    url = future_to_url[future]
                    try:
                        metadata = future.result()
                        metadata_results.append(metadata)
                    except Exception as exc:
                        print(f'URL {url} generated an exception: {exc}')
                        metadata_results.append({'url': url, 'error': str(exc)})

            # Convert metadata to DataFrame and merge
            metadata_df = pd.DataFrame(metadata_results)
            result_df = result_df.merge(metadata_df, left_on='SOURCEURL', right_on='url', how='left')

            # Drop duplicate url column
            if 'url' in result_df.columns:
                result_df = result_df.drop('url', axis=1)

        print(f"Found {len(result_df)} unique URLs from {len(filtered_df)} events")
        return result_df

    elif show_events:
        # Group by URL and get unique event descriptions and first date
        url_events = filtered_df.groupby('SOURCEURL').agg({
            'EventDescription': lambda x: list(x.dropna().unique()) if 'EventDescription' in filtered_df.columns else [],
            'SQLDATE': 'first'
        }).reset_index()

        # Convert SQLDATE to cosmograph-friendly format (YYYY-MM-DD)
        url_events['SQLDATE'] = pd.to_datetime(url_events['SQLDATE'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

        # Add event count for sorting
        url_events['event_count'] = url_events['EventDescription'].apply(len)
        url_events = url_events.sort_values('event_count', ascending=False)

        # Apply limit if specified
        if limit:
            url_events = url_events.head(limit)

        print(f"Found {len(filtered_df)} events with {len(url_events)} unique URLs")

        if extract_metadata:
            print(f"Extracting metadata for {len(url_events)} URLs...")

            def extract_metadata_with_context(row):
                time.sleep(delay)
                url, events, date = row['SOURCEURL'], row['EventDescription'], row['SQLDATE']
                metadata = extract_url_metadata(url, timeout)
                return (url, events, date, metadata)

            results_with_metadata = []

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(extract_metadata_with_context, row)
                          for _, row in url_events.iterrows()]

                for future in as_completed(futures):
                    try:
                        result = future.result()
                        results_with_metadata.append(result)
                    except Exception as exc:
                        print(f'Metadata extraction failed: {exc}')

            return results_with_metadata
        else:
            # Return list of tuples (url, events, date)
            return list(zip(url_events['SOURCEURL'], url_events['EventDescription'], url_events['SQLDATE']))

    else:
        # Original behavior - just URLs
        urls = filtered_df['SOURCEURL'].dropna().unique()

        if limit:
            urls = urls[:limit]

        print(f"Found {len(filtered_df)} events with {len(urls)} unique URLs")

        if extract_metadata:
            print(f"Extracting metadata for {len(urls)} URLs...")

            def extract_with_delay(url):
                time.sleep(delay)
                return (url, extract_url_metadata(url, timeout))

            results_with_metadata = []

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(extract_with_delay, url) for url in urls]

                for future in as_completed(futures):
                    try:
                        result = future.result()
                        results_with_metadata.append(result)
                    except Exception as exc:
                        print(f'Metadata extraction failed: {exc}')

            return results_with_metadata
        else:
            return urls.tolist()

# Additional utility function for analyzing metadata
def analyze_source_metadata(df_with_metadata):
    """
    Analyze the extracted metadata to provide insights
    """
    if 'site_name' not in df_with_metadata.columns:
        print("No metadata found in DataFrame")
        return

    print("=== Metadata Analysis ===")

    # Most common news sources
    print("\nTop news sources:")
    site_counts = df_with_metadata['site_name'].value_counts().head(10)
    for site, count in site_counts.items():
        if site:
            print(f"  {site}: {count} articles")

    # Language distribution
    if 'language' in df_with_metadata.columns:
        print("\nLanguage distribution:")
        lang_counts = df_with_metadata['language'].value_counts().head(5)
        for lang, count in lang_counts.items():
            if lang:
                print(f"  {lang}: {count} articles")

    # Success rate
    total_urls = len(df_with_metadata)
    successful = len(df_with_metadata[df_with_metadata['title'].notna()])
    print(f"\nMetadata extraction success rate: {successful}/{total_urls} ({successful/total_urls*100:.1f}%)")


import pandas as pd

def combine_multiple_columns(df, column_names, new_col_name='combined_text',
                           include_labels=True, separator='\n\n'):
    """
    Combine multiple text columns from a dataframe into a new column with formatted output.

    Parameters:
    -----------
    df : pd.DataFrame
        The input dataframe
    column_names : list
        List of column names to combine in order
    new_col_name : str
        Name for the new combined column (default: 'combined_text')
    include_labels : bool
        Whether to include column names as labels (default: True)
    separator : str
        Separator between text sections (default: '\n\n')

    Returns:
    --------
    pd.DataFrame
        The original dataframe with the new combined column added
    """

    # Create a copy to avoid modifying the original dataframe
    result_df = df.copy()

    # Validate column names
    for col_name in column_names:
        if col_name not in df.columns:
            raise ValueError(f"Column '{col_name}' not found in dataframe")

    def format_content(row):
        """Format the content for a single row"""
        parts = []

        for col_name in column_names:
            val = row[col_name]
            if pd.notna(val) and str(val).strip():
                if include_labels:
                    parts.append(f"{col_name}: {val}")
                else:
                    parts.append(str(val))

        # Join parts with separator
        return separator.join(parts) if parts else ""

    # Apply the formatting function to create the new column
    result_df[new_col_name] = df.apply(format_content, axis=1)

    return result_df

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


## Upload and parse

In [ ]:
from google.colab import files
import pandas as pd
import io

parser1 = False
parser2 = False

uploaded = files.upload()
for filename in uploaded.keys():
    print(f'User uploaded file "{filename}"')
    # Read the uploaded file into a pandas DataFrame
    try:
        data = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))
        print(f"DataFrame '{filename}' created successfully with shape: {data.shape}")
    except Exception as e:
        print(f"Error reading file {filename}: {e}")
        data = None

Saving Russia in africa tracker - Russia in africa tracker - Timeline w metadata (filled from news).csv to Russia in africa tracker - Russia in africa tracker - Timeline w metadata (filled from news).csv
User uploaded file "Russia in africa tracker - Russia in africa tracker - Timeline w metadata (filled from news).csv"
DataFrame 'Russia in africa tracker - Russia in africa tracker - Timeline w metadata (filled from news).csv' created successfully with shape: (1214, 21)


In [ ]:
data.head(1)

,SOURCEURL,Date,Event name,actor1_countries,actor2_countries,event_locations,combined_text_entities,combined_text_entity_types,keywords,first_event_date,...,description,event_count,avg_goldstein_score,site_name,language,actor1_names,actor2_names,event_descriptions,author,Type
0,https://www.africanews.com/2025/11/29/zumas-da...,11/29/2025,Zuma’s daughter resigns amid claims South Afri...,ZAF,RUS,SF,NaN,NaN,NaN,11/29/2025,...,The daughter of former South African President...,1.0,0.0,Africanews,en,SOUTH AFRICAN,RUSSIA,"['Use conventional military force, not specifi...",AfricaNews,SEC


In [ ]:
# Settings
output_filename = 'merged_doc_vectors_with_metadata.csv'

query = "Represent the event or story in the given news article based on the content provided"

#"Represent the event or story in the given news article based on the content provided"

# "Represent this segment of a position paper to understand the viewpoint expressed"

#"Represent the subject and sentiment of the instagram comment"

#"Represent the topic of the given comment based on the content provided"

# "Represent the themes and concepts in the given segment"

# Identify the topic or theme of the given news article based on the titles

text_field_1 ='title'
# 'title'
text_field_2 ='combined_text'
# 'combined_text' is default

cluster_dimensions=9
min_size = 5
min_samp = 1
eps = 0.22

meta_join = 'title'
# 'organization'

output_join = 'text_name' # text_name is default

## Processing with package

In [ ]:
from community_collection.package import *
if parser1:
  result_text = extract_text_content(df=data,
                      text_column='Content',
                      date_column='Date',
                      min_text_length = 5,
                      clean_whitespace = True)
  result_text.head(10)

if parser2:
  result = parse_message_content(
      df=data,
      url_column='Content',
      date_column='Date',
      delay=0,  # Shorter delay for example
      timeout=5
  )

else:
  result = None

In [ ]:
# data_to_process = data[(data['language'] == 'EN') & (data['feedback_text'].notna()) & (data['organization'].notna())].copy()

data_to_process = data

data_to_process = combine_multiple_columns(data_to_process, ['title','description'], new_col_name='combined_text',
                       include_labels=False, separator='\n\n')

# data_to_process = add_ner_columns(data_to_process,'body')

### Embedding process

In [ ]:
df_meta = data_to_process


In [ ]:
text_toprocess = data_to_process[text_field_2].tolist()

# Filter out any non-string values from the list
text_toprocess = [text for text in text_toprocess if isinstance(text, str)]

chunks = chunk(text_toprocess,
               model='intfloat/multilingual-e5-large-instruct',
               size=90)

doc_vectors, chunk_vectors = make_embedding(chunks,
    model="intfloat/multilingual-e5-large-instruct",
    instruction_prefix=query,
    return_chunk_vectors=True)

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

100%|██████████| 1372/1372 [00:21<00:00, 62.67 chunk/s]


### Create dataset

In [ ]:
import numpy as np
data = data_to_process

# Data filtering: Create a mask for valid document vectors
valid_doc_vector_indices = [i for i, vec in enumerate(doc_vectors) if not isinstance(vec, float)]

# Filter the DataFrame and the doc_vectors list based on the valid indices
filtered_data = data.iloc[valid_doc_vector_indices].copy()
doc_vecs_toprocess = [doc_vectors[i] for i in valid_doc_vector_indices]

# Now the texts and filenames are aligned with the filtered document vectors
doc_texts_toprocess = filtered_data[text_field_2].tolist()
filtered_filenames = filtered_data[text_field_1].tolist()


# Dimensionality reduction
clusterisable_vectors, map_vectors = reduce_vectors(doc_vecs_toprocess,clusterisable_dimensions=cluster_dimensions,map_dimensions=3)
save_reduced_vectors(clusterisable_vectors, map_vectors, "clusterisable_vectors.csv", "map_vectors.csv")

# Create DataFrame once
map_vectors_ = np.array(map_vectors)
vector_columns = [f"map_vector_{i}" for i in range(map_vectors_.shape[1])]
df_map_vectors = pd.DataFrame(map_vectors, columns=vector_columns)

# Add texts and save first version
df_map_vectors.insert(0, "original_text", doc_texts_toprocess)
df_map_vectors["text_name"] = filtered_filenames
df_map_vectors.to_csv('doc_vectors_original_text.csv')

# Create second version without original_text
df_map_vectors = pd.DataFrame(map_vectors, columns=vector_columns)
df_map_vectors["text_name"] = filtered_filenames
df_map_vectors.to_csv('doc_vectors.csv')

display(df_map_vectors.head())

X is normalized
PaCMAP(n_neighbors=10, n_MN=5, n_FP=20, distance=euclidean, lr=1.0, n_iters=(100, 100, 250), apply_pca=False, opt_method='adam', verbose=True, intermediate=False, seed=None)
Finding pairs
Found nearest neighbor
Calculated sigma
Found scaled dist
Pairs sampled successfully.
((12130, 2), (6065, 2), (24260, 2))
Initial Loss: 14942.0048828125
Iteration:   10, Loss: 11880.217773
Iteration:   20, Loss: 10040.865234
Iteration:   30, Loss: 9359.955078
Iteration:   40, Loss: 8924.785156
Iteration:   50, Loss: 8561.048828
Iteration:   60, Loss: 8197.906250
Iteration:   70, Loss: 7807.878906
Iteration:   80, Loss: 7365.633789
Iteration:   90, Loss: 6833.049805
Iteration:  100, Loss: 6104.434570
Iteration:  110, Loss: 7930.472656
Iteration:  120, Loss: 7875.462402
Iteration:  130, Loss: 7855.533691
Iteration:  140, Loss: 7849.780273
Iteration:  150, Loss: 7849.725586
Iteration:  160, Loss: 7849.203613
Iteration:  170, Loss: 7851.747559
Iteration:  180, Loss: 7852.945801
Iteration: 

,map_vector_0,map_vector_1,map_vector_2,text_name
0,8.309880,2.798806,0.549721,Zuma’s daughter resigns amid claims South Afri...
1,7.582633,3.275434,2.541942,South Africa arrests four men suspected of pla...
2,8.216564,2.772422,0.507918,S African ex-leader Zuma’s daughter quits parl...
3,7.690571,2.782805,1.074641,DA demands proper probe into Zuma-Sambudla’s a...
4,2.788325,-2.019564,-1.758027,Fire on Two Tankers After USV Attacks in Black...


## Visualization and clustering

### Clustering

In [ ]:
import pandas as pd
import hdbscan

# Merge to metadata

# Load the datasets
df_doc = pd.read_csv('/content/doc_vectors.csv')
# pd.read_csv('/content/GDELT RU Metadata.csv')

# Perform HDBSCAN clustering on clusterisable_vectors
# max_cluster_size = 800
clusterer = hdbscan.HDBSCAN(min_cluster_size=min_size,min_samples=min_samp, cluster_selection_epsilon=eps)
# Use the map_vectors from the correct dataframe (df_doc) for clustering
cluster_labels = clusterer.fit_predict(df_doc[['map_vector_0', 'map_vector_1', 'map_vector_2']].values)

# Add cluster labels to the df_doc dataframe
df_doc['cluster'] = cluster_labels

# Merge on id == title
merged = df_doc.merge(df_meta, left_on=output_join, right_on=meta_join, how='left')

# Save the full merged DataFrame for you to use
merged.to_csv(output_filename, index=False)

### Visualize (3d)

In [ ]:
import pandas as pd
import hdbscan
import plotly.express as px

# Load the merged data with correct cluster labels
df = pd.read_csv('/content/merged_doc_vectors_with_metadata.csv')

# The cluster labels are already in the dataframe from the previous step
# cluster_labels = clusterer.fit_predict(clustering_data)

# Add cluster labels to the dataframe
# df['cluster'] = cluster_labels # This is no longer needed as cluster is already in the dataframe

# Filter out noise points (cluster = -1) for plotting only
df_for_plot = df[df['cluster'] != -1].copy() # Add .copy() to avoid SettingWithCopyWarning

# Create the 3D scatter plot with cluster colors (excluding noise points)
fig = px.scatter_3d(df_for_plot, x='map_vector_0', y='map_vector_1', z='map_vector_2',
                    color=df_for_plot['cluster'].astype(str),
                    hover_data=[text_field_1],  # Replace 'column_name' with your desired column
                    color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_traces(marker=dict(size=2))
fig.update_layout(showlegend=False, margin=dict(l=10, r=10, t=10, b=10))

# Optional: Show cluster statistics
# Recalculate cluster labels from the filtered dataframe for correct counts
cluster_labels_for_stats = df['cluster'].values
print(f"Number of clusters: {len(set(cluster_labels_for_stats)) - (1 if -1 in cluster_labels_for_stats else 0)}")
print(f"Number of noise points: {list(cluster_labels_for_stats).count(-1)}")
print(f"Points plotted: {len(df_for_plot)} (excluding noise)")
print(f"Total points in dataset: {len(df)} (including noise)")
fig.show()

Number of clusters: 90
Number of noise points: 230
Points plotted: 1057 (excluding noise)
Total points in dataset: 1287 (including noise)


In [ ]:
# GDELT only
import pandas as pd

def reorder_and_filter_columns(df: pd.DataFrame, columns_to_keep: list) -> pd.DataFrame:
    """
    Reorder and/or drop columns in a pandas DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        The input dataframe.
    columns_to_keep : list
        List of columns to keep, in the desired order.
        Columns not in this list will be dropped.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns filtered and reordered.
    """
    # Filter the dataframe to only include columns in the list
    # Keep the order as provided
    return df.loc[:, [col for col in columns_to_keep if col in df.columns]]


df = reorder_and_filter_columns(df, ['first_event_date','avg_goldstein_score','title','description','cluster',
                                     'SOURCEURL','combined_text_entities', 'combined_text_entity_types','keywords','site_name','language',
                                     'event_count', 'actor1_names', 'actor2_names', 'actor1_countries', 'actor2_countries', 'event_locations',
                                     'event_descriptions', 'author', 'combined_text' ])

In [ ]:
df.to_csv('/content/RU_AFR_EXPERIMENT._3d.csv')

### Visualize (2D)

In [ ]:
import pandas as pd
import hdbscan
import plotly.express as px

# df = merged
# pd.read_csv('/content/vectorized_roundup_feed2.csv')

# Perform HDBSCAN clustering on the 2D coordinates (using only first two dimensions)
# clustering_data = df[['map_vector_0', 'map_vector_1','map_vector_2']].values
# clusterer = hdbscan.HDBSCAN(min_cluster_size=3, min_samples=3, cluster_selection_epsilon=0.48)
# cluster_labels = clusterer.fit_predict(clustering_data)

# # Add cluster labels to the dataframe
# df['cluster'] = cluster_labels

# # Filter out noise points (cluster = -1) for plotting only
# df_for_plot = df[df['cluster'] != -1]

# Create the 2D scatter plot with cluster colors (excluding noise points)
fig = px.scatter(df_for_plot, x='map_vector_0', y='map_vector_2',
                 color=df_for_plot['cluster'].astype(str),
                 hover_data=['text_name'],  # Replace 'text_name' with your desired column
                 color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_traces(marker=dict(size=4))  # Slightly larger markers for 2D visibility
fig.update_layout(showlegend=False,
                  margin=dict(l=10, r=10, t=10, b=10))
# Optional: Show cluster statistics
print(f"Number of clusters: {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)}")
print(f"Number of noise points: {list(cluster_labels).count(-1)}")
print(f"Points plotted: {len(df_for_plot)} (excluding noise)")
print(f"Total points in dataset: {len(df)} (including noise)")
fig.show()

Number of clusters: 90
Number of noise points: 220
Points plotted: 1057 (excluding noise)
Total points in dataset: 1287 (including noise)


## Document topics

### Load program

In [ ]:
!pip install -q bertopic

In [ ]:
import json
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
import re

def json_to_dataframe(json_path):
    """
    Converts a JSON file (with cluster, filenames, and content) into a pandas DataFrame.

    Parameters:
        - json_path (str): Path to the input JSON file.

    Returns:
        - DataFrame with columns: 'cluster', 'filename', 'content'.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    records = []
    for cluster, file_entries in data.items():
        for file_entry in file_entries:
            if isinstance(file_entry, dict):  # Ensure proper structure
                records.append({
                    "cluster": cluster,
                    "filename": file_entry["filename"],
                    "content": file_entry["content"]
                })

    return pd.DataFrame(records)


def use_bertopic_with_custom_vectors(chunk_vectors, documents, n_topics=7):
    """
    Apply BERTopic to custom vectors with full pipeline explicitly defined.
    """
    # Download NLTK stopwords if not already downloaded
    try:
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords')

    # Create an extended stopwords list
    english_stopwords = stopwords.words('english')
    additional_stopwords = []
    # [
    #     'maximum', 'characters', 'said', 'also', 'would', 'could', 'may',
    #     'might', 'like', 'many', 'much', 'get', 'well', 'even', 'still',
    #     'back', 'see', 'way', 'thing', 'make', 'made', 'got', 'go', 'going'
    # ]
    extended_stopwords = list(set(english_stopwords + additional_stopwords))

    # Preprocess documents
    preprocessed_documents = []
    for doc in documents:
        # Convert to lowercase
        text = doc.lower()
        # Remove special characters and normalize whitespace
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        preprocessed_documents.append(text)

    # 1. Convert vectors to proper numpy array
    embeddings = np.array(chunk_vectors)

    # 2. Define UMAP model for dimensionality reduction
    umap_model = UMAP(
        n_neighbors=15,       # Controls how local/global structure is preserved
        n_components=9,       # Reduced dimensions (5 is good for clustering)
        min_dist=0.01,         # Minimum distance between points in embedding
        metric='cosine',      # Distance metric
        random_state=42       # For reproducibility
        )

    # 3. Define HDBSCAN for clustering
    hdbscan_model = HDBSCAN(
        # max_cluster_size = 600,
        min_cluster_size=5,   # Matches our min_topic_size
        metric='euclidean',   # Distance metric for clustering
        cluster_selection_method='eom',  # Excess of mass (usually better)
        prediction_data=True, # Required for predicting on unseen data
        min_samples=1         # Controls cluster density
    )

    # 4. Define CountVectorizer for topic representation
    vectorizer = CountVectorizer(
        stop_words=extended_stopwords,
        ngram_range=(1, 2),
        min_df=0.05,
        max_df=0.90
    )

    # 5. Initialize BERTopic with full pipeline
    topic_model = BERTopic(
        embedding_model=None,           # We provide our own embeddings
        umap_model=umap_model,          # Explicitly use UMAP
        hdbscan_model=hdbscan_model,    # Explicitly use HDBSCAN
        vectorizer_model=vectorizer,    # Use our TF-IDF vectorizer
        nr_topics=n_topics,
        min_topic_size=15,
        top_n_words=15,
        calculate_probabilities=True,
        verbose=True
    )

    # 6. Fit the model with preprocessed documents and custom embeddings
    topics, probs = topic_model.fit_transform(preprocessed_documents, embeddings=embeddings)

    return topic_model, topics

### Execute

In [ ]:
# Usage in your workflow:
# After step 3 where you generate chunk_vectors
topic_model, topics = use_bertopic_with_custom_vectors(
    doc_vecs_toprocess,doc_texts_toprocess, n_topics = 2000)

# Add topics to your dataframe
result_df['bertopic_cluster'] = topics

2025-12-04 15:20:32,222 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-04 15:20:35,218 - BERTopic - Dimensionality - Completed ✓
2025-12-04 15:20:35,220 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-04 15:20:35,468 - BERTopic - Cluster - Completed ✓
2025-12-04 15:20:35,472 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-12-04 15:20:35,620 - BERTopic - Representation - Completed ✓
2025-12-04 15:20:35,622 - BERTopic - Topic reduction - Reducing number of topics
2025-12-04 15:20:35,626 - BERTopic - Topic reduction - Number of topics (2000) is equal or higher than the clustered topics(56).
2025-12-04 15:20:35,631 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-04 15:20:35,760 - BERTopic - Representation - Completed ✓


In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,126,-1_2025_six_african_federation,"[2025, six, african, federation, tanzania, cou...",[ukraine russia says advances made in key city...
1,0,6,0_release_conversation_kenyans_detained,"[release, conversation, kenyans, detained, tel...",[too little too late why ruto has failed kenya...
2,1,10,1_nigeria_action_trump_possible,"[nigeria, action, trump, possible, donald, pre...",[russia monitoring potential us military actio...
3,2,7,2_alabuga_zimbabwe_cameroon_program,"[alabuga, zimbabwe, cameroon, program, concern...",[alabuga migrant battalion zambia s first cont...
4,3,8,3_alabuga_programme_program_investigation,"[alabuga, programme, program, investigation, s...",[alabagu start programme russia influencer ala...
5,4,10,4_el_egypt_vessel_nuclear,"[el, egypt, vessel, nuclear, power, plant, pow...",[satellites show new russian built nuclear pow...
6,5,22,5_nuclear_ethiopia_nuclear power_plant,"[nuclear, ethiopia, nuclear power, plant, powe...",[russia and ethiopia sign action plan for nucl...
7,6,14,6_lavrov_kremlin_foreign minister_sergei,"[lavrov, kremlin, foreign minister, sergei, tr...",[putin sends sergei lavrov into disgrace after...
8,7,5,7_naval_china_russia china_brics,"[naval, china, russia china, brics, set, joint...",[exercise mosi iii set down for november defen...
9,8,7,8_brics_summit_future_moscow,"[brics, summit, future, moscow, sa, participat...",[south africa to attend brics fashion summit i...


In [ ]:
topic_model.get_topic_info().to_csv('document_topics.csv')

In [ ]:
df_map_vectors['bertopic_cluster'] = topics
if 'original_text' in df_map_vectors.columns:
    df_map_vectors.drop('original_text', axis=1, inplace=True)
df_map_vectors.insert(0, "original_text", doc_texts_toprocess)  # Add original texts

df_map_vectors.to_csv('doc_vectors.csv')

In [ ]:
# Load the original dataframe
df_wuefa = pd.read_csv('/content/WUEFA EN Comments Merged Doc Vectors with Metadata._3d.csv')

# Load the dataframe with bertopic cluster labels
df_bertopic_clusters = pd.read_csv('doc_vectors.csv')

# Merge the two dataframes on a common column (assuming 'text_name' and 'body' can be used for merging)
# You might need to adjust the 'left_on' and 'right_on' based on your actual data
merged_df = df_wuefa.merge(df_bertopic_clusters[['original_text', 'bertopic_cluster']],
                           left_on='title', # Assuming 'body' in wuefa df matches 'text_name' in doc_vectors
                           right_on='original_text',
                           how='left')

# Drop the redundant 'text_name' column from the merged dataframe
# merged_df.drop('text_name', axis=1, inplace=True)

# Save the updated dataframe to a new CSV
merged_df.to_csv('WUEFA_EN_Comments_Merged_Doc_Vectors_with_Metadata_and_Topics.csv', index=False)

print("Updated CSV saved with BERTopic cluster labels.")

Updated CSV saved with BERTopic cluster labels.


In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.get_topic(2)

[('alabuga', np.float64(0.7000430308370995)),
 ('zimbabwe', np.float64(0.25420273050415976)),
 ('cameroon', np.float64(0.1803942475504149)),
 ('program', np.float64(0.17345002643322702)),
 ('concerns', np.float64(0.16193147424616167)),
 ('kenya', np.float64(0.11826606403642735)),
 ('uganda', np.float64(0.11265191824519626)),
 ('16', np.float64(0.1125426459523453)),
 ('cases', np.float64(0.1125426459523453)),
 ('govt', np.float64(0.1125426459523453)),
 ('recruitment', np.float64(0.09929816544952302)),
 ('left', np.float64(0.09901517962512559)),
 ('looking', np.float64(0.09901517962512559)),
 ('investigation', np.float64(0.0661396303190582)),
 ('trafficking', np.float64(0.06480798034590192))]

In [ ]:
result_df.to_csv('chunks_with_coordinates.csv')

NameError: name 'result_df' is not defined

## Network analysis

#### Functions and packges

In [ ]:
!pip install cosmograph -q
!pip install cosmograph_widget -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.9/278.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.1/256.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import pandas as pd
import ast
from itertools import combinations
import cosmograph

def parse_to_cosmograph(
    filepath: str,
    entities_col: str,
    types_col: str,
    desired_entity_types: list,
    timestamp_col: str = 'event_date'
) -> pd.DataFrame:
    """
    Parses a CSV and returns co-occurrence edges for specified entity types,
    preserving all other metadata and converting timestamps to ISO 8601.

    Parameters
    ----------
    filepath : str
        Path to the input CSV file.
    entities_col : str
        Column name with list of entity strings (as Python list string).
    types_col : str
        Column name with list of corresponding entity types.
    desired_entity_types : list
        List of entity types to include in edge generation.
    timestamp_col : str
        Column name containing timestamp or date values.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
          - source
          - target
          - all original metadata columns (with timestamp converted to ISO 8601)
    """
    # Load data
    df = pd.read_csv(filepath)

    # Identify metadata columns (all except the entity and type lists)
    metadata_cols = [c for c in df.columns if c not in (entities_col, types_col)]

    edges = []
    for _, row in df.iterrows():
        # Parse entities and types
        try:
            ents = ast.literal_eval(row[entities_col])
            types = ast.literal_eval(row[types_col])
        except (ValueError, SyntaxError):
            continue

        # Filter entities by desired types
        filtered = [(ent, typ) for ent, typ in zip(ents, types) if typ in desired_entity_types]
        if len(filtered) < 2:
            continue

        # Format metadata for this row and convert timestamp
        meta = {col: row[col] for col in metadata_cols}
        if timestamp_col in meta:
            dt = pd.to_datetime(meta[timestamp_col], errors='coerce')
            meta[timestamp_col] = dt.strftime('%Y-%m-%dT%H:%M:%SZ') if not pd.isna(dt) else None

        # Create edges for each combination
        for (ent1, _), (ent2, _) in combinations(filtered, 2):
            edge = {'source': ent1, 'target': ent2}
            edge.update(meta)
            edges.append(edge)

    return pd.DataFrame(edges)


def network_stats(
    edges_df: pd.DataFrame,
    source: str = 'source',
    target: str = 'target',
    top_n: int = 10
) -> None:
    """
    Prints basic network statistics:
      - Total nodes
      - Total edges
      - Average degree
      - Entity type distribution (if type columns exist)
      - Top N most connected entities
      - Top N most common source-target pairs
    """
    # Basic stats
    total_edges = len(edges_df)
    nodes = pd.concat([edges_df[source], edges_df[target]]).unique()
    total_nodes = len(nodes)
    avg_degree = round(2 * total_edges / total_nodes, 2) if total_nodes else 0

    # Degree counts
    degree_counts = pd.concat([edges_df[source], edges_df[target]]).value_counts()

    print("Network Statistics:")
    print(f"- Total nodes: {total_nodes}")
    print(f"- Total edges: {total_edges}")
    print(f"- Average degree: {avg_degree}\n")

    print(f"Top {top_n} Most Connected Entities:")
    for entity, cnt in degree_counts.head(top_n).items():
        print(f"- {entity}: {cnt} connections")

    # Top source-target pairs
    pair_counts = edges_df.groupby([source, target]).size().sort_values(ascending=False)
    print(f"\nTop {top_n} Most Common Pairs:")
    for (src, tgt), cnt in pair_counts.head(top_n).items():
        print(f"- {src} <-> {tgt}: {cnt} occurrences")



#### Execution

In [ ]:
edge_data = parse_to_cosmograph('/content/WUEFA EN Comments Merged Doc Vectors with Metadata._3d.csv',
                    'body','body_entity_types', ['PERSON', 'ORG', 'NORP', 'LOC', 'GPE'])
edge_data.to_csv('edge_data_w_uefa.csv')

In [ ]:
network_stats(edge_data)

KeyError: 'source'

# Old

In [ ]:
import numpy as np
data = data_to_process

# Data filtering: Create a mask for valid document vectors
valid_doc_vector_indices = [i for i, vec in enumerate(doc_vectors) if not isinstance(vec, float)]

# Filter the DataFrame and the doc_vectors list based on the valid indices
filtered_data = data.iloc[valid_doc_vector_indices].copy()
doc_vecs_toprocess = [doc_vectors[i] for i in valid_doc_vector_indices]

# Now the texts and filenames are aligned with the filtered document vectors
doc_texts_toprocess = filtered_data[text_field_2].tolist()
filtered_filenames = filtered_data[text_field_1].tolist()


# Dimensionality reduction
clusterisable_vectors, map_vectors = reduce_vectors(doc_vecs_toprocess,map_dimensions=map_dimensions)
save_reduced_vectors(clusterisable_vectors, map_vectors, "clusterisable_vectors.csv", "map_vectors.csv")

# Create DataFrame once
map_vectors_ = np.array(map_vectors)
vector_columns = [f"map_vector_{i}" for i in range(map_vectors_.shape[1])]
df_map_vectors = pd.DataFrame(map_vectors, columns=vector_columns)

# Add texts and save first version
df_map_vectors.insert(0, "original_text", doc_texts_toprocess)
df_map_vectors["text_name"] = filtered_filenames
df_map_vectors.to_csv('doc_vectors_original_text.csv')

# Create second version without original_text
df_map_vectors = pd.DataFrame(map_vectors, columns=vector_columns)
df_map_vectors["text_name"] = filtered_filenames
df_map_vectors.to_csv('doc_vectors.csv')

display(df_map_vectors.head())

In [ ]:
import spacy

nlp = spacy.load("en_core_web_trf")
doc = nlp("This is a simple sentence with a great big noun phrase.")
print("has parser:", nlp.has_pipe("parser"))
print([nc.text for nc in doc.noun_chunks])  # should show chunks


has parser: True
['This', 'a simple sentence', 'a great big noun phrase']
